In [ ]:
# 1 extraction features TF-IDF
import pandas as pd
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

print("🔄 [TAHAP 1] Memulai Preprocessing & TF-IDF...")

# --- CONFIG ---
# Gunakan Absolute Path dengan 'r' di depan agar aman di Windows
FILE_DATA = r'd:\project skripsi machine learning intoleransi\virtualEnvironment\dataset\dataYangDiPakai\data_label_revisi_goblog.csv' 

# 0. CEK KEAMANAN FILE SEBELUM JALAN
if not os.path.exists(FILE_DATA):
    raise FileNotFoundError(f"❌ File tidak ditemukan di jalur:\n{FILE_DATA}\nCoba pastikan nama file dan foldernya sudah persis sama!")

# Bikin folder otomatis
os.makedirs('output', exist_ok=True) 
os.makedirs('models', exist_ok=True) 

# 1. LOAD DATA
print("   - Membaca dataset...")
df = pd.read_csv(FILE_DATA, sep=';') 

# Cek keamanan kolom (Biar tidak error kalau nama kolom salah)
if 'terjemahan_indo' not in df.columns or 'label' not in df.columns:
    print(f"Daftar kolom yang ada di filemu: {df.columns.tolist()}")
    raise KeyError("❌ Kolom 'terjemahan_indo' atau 'label' tidak ada! Coba cek tulisan di atas, pastikan namanya cocok.")

# Bersihkan data kosong
df = df.dropna(subset=['terjemahan_indo', 'label']) 
print(f"   - Total data bersih yang siap diproses: {len(df)} baris")

# 2. TF-IDF (Ubah Huruf jadi Angka)
print("   - Melakukan ekstraksi fitur TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['terjemahan_indo'].astype(str))
y = df['label'].astype(int) # Pastikan label berupa angka (0, 1, 2)

# 3. SIMPAN KAMUS TF-IDF
joblib.dump(vectorizer, 'models/vectorizer_tfidf.pkl')

# 4. SPLIT DATA (80% Latih, 20% Uji)
print("   - Memecah data (80% Data Latih, 20% Data Uji)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. SIMPAN DATA MATANG
print("   - Menyimpan data matang ke folder 'output'...")
joblib.dump(X_train, 'output/X_train.pkl')
joblib.dump(X_test,  'output/X_test.pkl')
joblib.dump(y_train, 'output/y_train.pkl')
joblib.dump(y_test,  'output/y_test.pkl')

print("✅ SELESAI TAHAP 1. Data sudah siap!")
print("👉 Silakan lanjut jalankan '2_training.py' atau '2_training.ipynb'")

In [ ]:
# 2 pelatihan model SVM, KNN dan ensemble
import joblib
import pandas as pd
import os
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV

print("🏋️ [TAHAP 2] Training: DATA ASLI (Tanpa Penyeimbang Apapun)...")

# 0. CEK KEAMANAN: Pastikan folder dan file Tahap 1 sudah ada
if not os.path.exists('output/X_train.pkl'):
    raise FileNotFoundError("❌ File 'output/X_train.pkl' tidak ditemukan! Pastikan kamu sudah menjalankan '1_preprocessing.py' terlebih dahulu.")
os.makedirs('models', exist_ok=True) # Jaga-jaga kalau folder models terhapus

# 1. AMBIL DATA DARI TAHAP 1
print("   - Memuat data latih...")
X_train = joblib.load('output/X_train.pkl')
y_train = joblib.load('output/y_train.pkl')

print(f"   - Jumlah Data Latih Asli: {len(y_train)} baris")
print(f"   - Komposisi Label: {y_train.value_counts().to_dict()}")

# ---------------------------------------------------------
# 2. LATIH SVM (MODEL UTAMA)
# ---------------------------------------------------------
print("\n🚀 Melatih SVM (Mencari Settingan Terbaik)...")
print("   (Mohon tunggu, ini akan memakan waktu beberapa menit ☕)")

param_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# n_jobs=-1 artinya kita memakai seluruh "otak" CPU laptop agar cepat selesai
svm_grid = GridSearchCV(SVC(probability=True, random_state=42), param_svm, cv=3, verbose=1, n_jobs=-1)
svm_grid.fit(X_train, y_train) 

best_svm = svm_grid.best_estimator_
joblib.dump(best_svm, 'models/model_svm.pkl')
print(f"   ✅ SVM Selesai (Akurasi Validasi: {svm_grid.best_score_*100:.2f}%)")

# ---------------------------------------------------------
# 3. LATIH KNN (METRIC COSINE)
# ---------------------------------------------------------
print("\n🚀 Melatih KNN (Wajib Cosine)...")

param_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15], 
    'metric': ['cosine'] 
}

knn_grid = GridSearchCV(KNeighborsClassifier(), param_knn, cv=3, verbose=1, n_jobs=-1)
knn_grid.fit(X_train, y_train)

best_knn = knn_grid.best_estimator_
joblib.dump(best_knn, 'models/model_knn.pkl')
print(f"   ✅ KNN Selesai (Best K: {knn_grid.best_params_['n_neighbors']})")

# ---------------------------------------------------------
# 4. LATIH ENSEMBLE (SVM + KNN)
# ---------------------------------------------------------
print("\n🚀 Melatih ENSEMBLE (Voting SVM + KNN)...")

# Gabungkan dua model terbaik
ensemble_model = VotingClassifier(
    estimators=[
        ('svm', best_svm), 
        ('knn', best_knn)
    ],
    voting='soft',
    weights=[2, 1] # SVM kita beri bobot suara lebih tinggi
)

ensemble_model.fit(X_train, y_train)
joblib.dump(ensemble_model, 'models/model_ensemble.pkl')
print("   ✅ Ensemble Selesai.")

print("\n==================================================")
print("🎉 TRAINING DATA MURNI SELESAI!")
print("👉 Silakan jalankan '3_evaluasi.py' untuk melihat hasil akhirnya.")

In [ ]:
# 3 evaluasi hasil model
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os

print("📊 [TAHAP 3] Evaluasi Hasil Model...")

# 0. CEK KEAMANAN
if not os.path.exists('output/X_test.pkl'):
    raise FileNotFoundError("❌ File 'output/X_test.pkl' tidak ditemukan! Pastikan Tahap 1 sudah dijalankan.")
if not os.path.exists('models/model_svm.pkl'):
    raise FileNotFoundError("❌ File 'models/model_svm.pkl' tidak ditemukan! Pastikan Tahap 2 sudah selesai 100%.")

# Bikin folder images otomatis
os.makedirs('images', exist_ok=True)

# 1. Ambil Data Uji
print("   - Memuat data uji...")
X_test = joblib.load('output/X_test.pkl')
y_test = joblib.load('output/y_test.pkl')

# 2. Daftar Model (CUMA 3 SEKARANG)
print("   - Memuat model-model AI...")
daftar_model = {
    "SVM": joblib.load('models/model_svm.pkl'),
    "KNN": joblib.load('models/model_knn.pkl'),
    "Ensemble": joblib.load('models/model_ensemble.pkl')
}

# 3. Loop Evaluasi
for nama, model in daftar_model.items():
    print(f"\n==========================================")
    print(f"--- Evaluasi Model: {nama} ---")
    print(f"==========================================")

    # Lakukan Prediksi
    y_pred = model.predict(X_test)
    
    # Hitung Akurasi
    acc = accuracy_score(y_test, y_pred)
    print(f"🎯 Akurasi {nama}: {acc*100:.2f}%\n")
    
    # Laporan Lengkap (Precision, Recall, F1-Score)
    print("📋 Laporan Klasifikasi:")
    print(classification_report(y_test, y_pred))

    # Bikin Grafik Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                annot_kws={"size": 14}) # Angka di dalam kotak diperbesar
    
    # Hiasan Grafik (Lebih rapi untuk masuk ke buku Skripsi)
    plt.title(f'Confusion Matrix - {nama}\n(Akurasi: {acc*100:.2f}%)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Prediksi Mesin', fontsize=12, fontweight='bold')
    plt

In [ ]:
# 3.2 Menampilkan Distribusi Data
# Menampilkan perbandingan jumlah data dan tampilan grafik perbandingan data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Pastikan folder images ada untuk menyimpan hasil
os.makedirs('images', exist_ok=True)

# ==========================================
# 1. LOAD DATA
# ==========================================
nama_file = r'd:\project skripsi machine learning intoleransi\virtualEnvironment\dataset\dataYangDiPakai\data_label_revisi_goblog.csv'

print(f"📂 Membaca file: {nama_file}...")
df = pd.read_csv(nama_file, sep=';') 

# ==========================================
# 2. HITUNG JUMLAH LABEL
# ==========================================
# Kita pastikan kolom 'label' ada
if 'label' not in df.columns:
    raise ValueError("❌ Kolom 'label' tidak ditemukan di dataset! Coba cek separatornya (sep=';' atau sep=',')")

jumlah_label = df['label'].value_counts().sort_index()

print("\n📊 STATISTIK JUMLAH DATA:")
print("-" * 30)
label_names = {0: "Netral (0)", 1: "Kritik (1)", 2: "Hujatan (2)"}
for lbl, count in jumlah_label.items():
    print(f"   {label_names.get(lbl, lbl)}: {count} data")
print("-" * 30)
print(f"   TOTAL: {len(df)} data")

# ==========================================
# 3. BUAT GRAFIK (VISUALISASI)
# ==========================================
plt.figure(figsize=(8, 6)) # Ukuran gambar (Lebar, Tinggi)

# Bikin Bar Chart warna-warni (tambah hue=... agar tidak muncul warning di versi Seaborn terbaru)
ax = sns.barplot(x=jumlah_label.index, y=jumlah_label.values, hue=jumlah_label.index, palette='viridis', legend=False)

# Hiasan Grafik
plt.title('Perbandingan Jumlah Data per Label', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Kategori Label', fontsize=12, fontweight='bold')
plt.ylabel('Jumlah Data', fontsize=12, fontweight='bold')

# Pastikan urutan label sesuai dengan 0, 1, 2
urutan_label = sorted(jumlah_label.index.tolist())
plt.xticks(ticks=range(len(urutan_label)), labels=['0\n(Netral)', '1\n(Kritik)', '2\n(Hujatan)'])
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Tampilkan Angka di Atas Batang
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 10), 
                textcoords='offset points',
                fontsize=14, fontweight='bold', color='black')

# Simpan Gambar di dalam folder 'images'
nama_gambar = 'images/grafik_distribusi_data.png'
plt.savefig(nama_gambar, dpi=300, bbox_inches='tight')
print(f"\n🖼️ Grafik berhasil disimpan: {nama_gambar}")

# Tampilkan gambar di layar
plt.show()

In [ ]:
import joblib
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV

print("🔬 [EKSPERIMEN] Membandingkan 3 Metode Mencari Nilai K Terbaik...")

# 0. CEK KEAMANAN
if not os.path.exists('output/X_train.pkl'):
    raise FileNotFoundError("❌ File 'output/X_train.pkl' tidak ditemukan! Pastikan Tahap 1 sudah dijalankan.")

# Pastikan folder images ada
os.makedirs('images', exist_ok=True)

# 1. LOAD DATA
X_train = joblib.load('output/X_train.pkl')
y_train = joblib.load('output/y_train.pkl')
jumlah_data = X_train.shape[0]

print(f"   - Jumlah Data Latih: {jumlah_data} baris")
print("-" * 50)

results = [] # Untuk menyimpan hasil perbandingan

# =========================================================
# METODE 1: AKAR KUADRAT (Square Root Rule)
# Rumus: K = Akar(Total Data)
# =========================================================
print("1️⃣ Menguji Metode Akar Kuadrat...")
k_sqrt = int(math.sqrt(jumlah_data))

# Aturan: K harus ganjil biar gak seri (draw)
if k_sqrt % 2 == 0:
    k_sqrt += 1

# Uji Akurasinya (Pakai n_jobs=-1 biar ngebut)
knn_sq = KNeighborsClassifier(n_neighbors=k_sqrt, metric='cosine')
scores_sq = cross_val_score(knn_sq, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
acc_sq = scores_sq.mean() * 100

print(f"   -> Hasil: K={k_sqrt}, Akurasi={acc_sq:.2f}%")
results.append({'Metode': 'Akar Kuadrat', 'K': k_sqrt, 'Akurasi': acc_sq})

# =========================================================
# METODE 2: ELBOW METHOD (Metode Siku)
# Coba manual dari 1 sampai 40, lalu cari error terkecil
# =========================================================
print("\n2️⃣ Menguji Metode Elbow (Looping 1-40)...")
print("   (Tunggu sebentar, sedang menghitung manual...)")
error_rates = []
acc_rates = []
k_range = range(1, 41, 2) # Coba angka ganjil: 1, 3, 5, ... 39

best_k_elbow = 0
best_acc_elbow = 0

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    # Pakai n_jobs=-1 di sini juga biar loopingnya gak kelamaan
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    acc = scores.mean()

    # Simpan data buat grafik
    acc_rates.append(acc)
    error_rates.append(1 - acc) # Error = 100% - Akurasi

    # Cek apakah ini rekor terbaik?
    if acc > best_acc_elbow:
        best_acc_elbow = acc
        best_k_elbow = k

print(f"   -> Hasil Terbaik di Range Ini: K={best_k_elbow}, Akurasi={best_acc_elbow*100:.2f}%")
results.append({'Metode': 'Elbow (Manual)', 'K': best_k_elbow, 'Akurasi': best_acc_elbow*100})

# Bikin Grafik Elbow
plt.figure(figsize=(10, 6))
plt.plot(k_range, error_rates, color='red', linestyle='dashed', marker='o',
         markerfacecolor='blue', markersize=8)
plt.title('Grafik Elbow (Mencari Error Terkecil)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Nilai K', fontsize=12, fontweight='bold')
plt.ylabel('Tingkat Error', fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# Simpan ke folder images
nama_gambar_elbow = 'images/grafik_elbow_knn.png'
plt.savefig(nama_gambar_elbow, dpi=300, bbox_inches='tight')
print(f"   🖼️ Grafik Elbow tersimpan: {nama_gambar_elbow}")
plt.close() # Tutup grafik biar memori lega

# =========================================================
# METODE 3: GRID SEARCH CV (Validasi Silang)
# Ini metode paling 'Sultan' dan Valid
# =========================================================
print("\n3️⃣ Menguji Metode Grid Search CV (Otomatis)...")
param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15, 19, 21, 25, 29]}
grid = GridSearchCV(KNeighborsClassifier(metric='cosine'), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

k_grid = grid.best_params_['n_neighbors']
acc_grid = grid.best_score_ * 100

print(f"   -> Hasil: K={k_grid}, Akurasi={acc_grid:.2f}%")
results.append({'Metode': 'Grid Search CV', 'K': k_grid, 'Akurasi': acc_grid})

# =========================================================
# KESIMPULAN AKHIR
# =========================================================
print("\n" + "="*50)
print("🏆 TABEL PERBANDINGAN METODE PENENTUAN K")
print("="*50)
df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))
print("-" * 50)

# Cari pemenang
best_method = df_res.loc[df_res['Akurasi'].idxmax()]
print(f"✅ REKOMENDASI: Gunakan K = {best_method['K']}")
print(f"   (Berdasarkan metode {best_method['Metode']} dengan akurasi tertinggi)")